# Karatsuba Multiplication for Polynomials / Big Numbers

This notebook implements the **Karatsuba algorithm**, a divide-and-conquer method for multiplying two sequences of coefficients (polynomials, or the digit arrays of large integers).

**The idea:** schoolbook multiplication of two length-`n` sequences takes `n²` single multiplications. Karatsuba reduces this to roughly `n^1.585` by replacing 4 recursive half-size multiplications with only 3.

**Representation used here:** a NumPy array `[c0, c1, c2, ...]` represents the polynomial
`c0 + c1·x + c2·x² + ...` (lowest degree first). Multiplying two arrays is therefore the same as multiplying the polynomials, which is also a *convolution* of the arrays.

**Notebook outline**
1. Import libraries
2. `split`: cut an array into a low half and a high half
3. `multiply`: the recursive Karatsuba routine
4. Test on two example polynomials
5. Results and complexity analysis

## 1. Imports

We only need **NumPy**. Its arrays give us fast slicing, element-wise addition/subtraction, and scalar multiplication (`a*b` when one operand has size 1), which the algorithm relies on.

In [5]:
import numpy as np

## 2. Helper: `split(a)`

Splits an array into two halves at the midpoint:

- `a0 = a[0:mid]`: the **low-order** coefficients
- `a1 = a[mid:]`: the **high-order** coefficients
- `mid = a.size // 2`: also returned, because it is the power of `x` by which the high half is shifted

So `a(x) = a0(x) + x^mid · a1(x)`.

**Example:** `split([1, 5, 32, 1])` → `a0 = [1, 5]`, `a1 = [32, 1]`, `mid = 2`.

Slicing returns NumPy views, so no data is copied here.

In [6]:
def split(a):
    mid = a.size//2
    a0 = a[0:mid]
    a1 = a[mid:a.size]
    return a0,a1,mid

## 3. Core: `karatsuba(a, b)`

### Math behind it
Split both inputs: `a = a0 + x^m·a1` and `b = b0 + x^m·b1`. Then

```
a·b = a0b0 + x^m · (a0b1 + a1b0) + x^(2m) · a1b1
```

This looks like **4** sub-multiplications, but the Karatsuba trick gets the middle term from one:

```
a0b1 + a1b0 = (a0 + a1)(b0 + b1) − a0b0 − a1b1
```

so only **3** recursive multiplications are needed: `a0b0`, `a1b1`, and `(a0+a1)(b0+b1)`.

### Step by step
1. **Base case:** if either array has size 1, return `a*b`. This is plain scalar multiplication through NumPy broadcasting.
2. **Split** `a` and `b` into low/high halves. `power` is the shift `m`.
3. **Recurse** to get `a0b0` and `a1b1`.
4. **Compute the middle term** with the third recursive call, then subtract `a0b0` and `a1b1`.
5. **Combine:** allocate `result` of length `n + m − 1`, the exact length of a polynomial product, then add each piece at its shifted offset:
   - `a0b0` at offset `0`
   - `middle` at offset `power`
   - `a1b1` at offset `2·power`

   Overlapping regions are *added* (`+=`), which handles carries between terms.

### Assumptions and limitations
- Designed for **equal-length inputs whose size is a power of two** (as in the test below). `power` is taken from `a` only, so unequal or odd sizes may give wrong results.
- `dtype=int` is used for the result. Very large coefficients could overflow int64.

In [ ]:
def karatsuba(a,b):
    n = a.size
    m = b.size
    if(n==1 or m==1):
        return a*b
    a0,a1,power = split(a)
    b0,b1,temp  = split(b)


    a0b0 = karatsuba(a0,b0)
    a1b1 = karatsuba(a1,b1)


    middle = karatsuba((a0+a1),(b0+b1)) - a0b0 - a1b1
    result = np.zeros(n+m-1,dtype =int)
    result[:a0b0.size] += a0b0
    result[power:power+middle.size]+=middle
    result[2*power : 2*power+a1b1.size]+=a1b1

    return result

## 4. Test

Two 8-element polynomials (zero-padded to a power of two):

- `a(x) = 1 + 5x + 32x² + x³`
- `b(x) = 1 + 3x³ + 45x⁴ + 2x⁵`

The product has `8 + 8 − 1 = 15` coefficients. We call `Karatsuba(a, b)` and print them.

In [ ]:
a = np.array([1, 5, 32, 1, 0, 0, 0 ,0])
b = np.array([1, 0, 0, 3, 45, 2, 0 ,0])

print(f"a: {a}")
print(f"b: {b}")

print(f"Multiplied polynomial: {karatsuba(a,b)}")

a: [ 1  5 32  1  0  0  0  0]
b: [ 1  0  0  3 45  2  0  0]
Multiplied polynomial: [   1    5   32    4   60  323 1453  109    2    0    0    0    0    0
    0]


## 5. Results

**Output**
```
[   1    5   32    4   60  323 1453  109    2    0    0    0    0    0    0]
```

### Verification (by hand)
Multiplying `(1 + 5x + 32x² + x³)(1 + 3x³ + 45x⁴ + 2x⁵)` term by term:

| Power | Coefficient | Computation |
|---|---|---|
| x⁰ | 1 | 1·1 |
| x¹ | 5 | 5·1 |
| x² | 32 | 32·1 |
| x³ | 4 | 1·1 + 1·3 |
| x⁴ | 60 | 5·3 + 1·45 |
| x⁵ | 323 | 32·3 + 5·45 + 1·2 |
| x⁶ | 1453 | 1·3 + 32·45 + 5·2 |
| x⁷ | 109 | 1·45 + 32·2 |
| x⁸ | 2 | 1·2 |
| x⁹–x¹⁴ | 0 | padding zeros |

This matches the program output, and `np.convolve(a, b)` returns the same array.

## 6. Complexity Analysis

### Time complexity
Each call does 3 recursive multiplications on inputs of half the size, plus O(n) work for splitting, adding, subtracting, and combining:

```
T(n) = 3·T(n/2) + O(n)
```

By the Master Theorem (a = 3, b = 2, so n^(log₂3) dominates):

**T(n) = O(n^log₂3) ≈ O(n^1.585)**

| Method | Time |
|---|---|
| Schoolbook | O(n²) |
| Karatsuba (this notebook) | O(n^1.585) |

For small `n` the overhead of recursion and array allocation makes Karatsuba slower in practice than a direct method (such as `np.convolve`). It pays off for large inputs.

### Space complexity
- Each level allocates temporary arrays (`a0+a1`, `b0+b1`, `middle`, `result`) of size O(n).
- The recursion depth is O(log n), and the input size halves at each level, so the temporaries form a geometric series: n + n/2 + n/4 + … = O(n).

**Auxiliary space: O(n)**, plus the O(n) output array.

### Summary
| | Complexity |
|---|---|
| Time | O(n^log₂3) ≈ O(n^1.585) |
| Space | O(n) |
| Recursion depth | O(log n) |